# CareerOps-4B private merge (LoRA → full)
Keeps HF **private**. Needs dataset `careerops-runtime-auth` (`hf_token`).
Uploads merged weights to private `telivity/CareerOps-4B-merged`.

In [ ]:
!pip -q install -U "transformers>=4.51" peft accelerate huggingface_hub

In [ ]:
import os, glob
from pathlib import Path

auth = sorted(Path('/kaggle/input').glob('**/hf_token'))
assert auth, 'attach private careerops-runtime-auth'
os.environ['HF_TOKEN'] = Path(auth[0]).read_text().strip()
assert len(os.environ['HF_TOKEN']) > 10
print('HF_TOKEN: present')

BASE = 'google/gemma-4-E2B-it'
ADAPTER = 'telivity/CareerOps-4B'
OUT_REPO = 'telivity/CareerOps-4B-merged'
OUT = Path('/kaggle/working/merged')
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import HfApi, create_repo

tok = os.environ['HF_TOKEN']
print('loading base on CPU…', flush=True)
tokenizer = AutoTokenizer.from_pretrained(BASE, token=tok)
model = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map='cpu', token=tok,
    low_cpu_mem_usage=True)
print('loading adapter…', flush=True)
model = PeftModel.from_pretrained(model, ADAPTER, token=tok)
print('merge_and_unload…', flush=True)
model = model.merge_and_unload()
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print('saved', OUT, flush=True)

create_repo(OUT_REPO, private=True, exist_ok=True, token=tok)
api = HfApi(token=tok)
api.upload_folder(
    folder_path=str(OUT), repo_id=OUT_REPO, token=tok,
    commit_message='private merged full weights from CareerOps-4B adapter')
info = api.repo_info(OUT_REPO, repo_type='model')
assert info.private is True, 'refusing non-private upload'
print('uploaded PRIVATE', OUT_REPO, flush=True)